# Demo — Full Decoding Pipeline Walkthrough

End-to-end demonstration of the BCI decoding pipeline: generate a synthetic
neural signal, preprocess it, run model inference, decode with greedy search,
beam search, and LM correction, then visualize CTC probabilities and the
final decoded text.

## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from src.config import get_default_config
from src.preprocessing.filter import bandpass_filter, notch_filter
from src.preprocessing.normalize import zscore_normalize
from src.models.gru_decoder import GRUDecoder
from src.decoding.greedy import greedy_decode
from src.decoding.beam_search import beam_search_decode
from src.decoding.lm_correction import lm_rescore
from src.visualization.ctc_plots import plot_ctc_heatmap
from src.visualization.signal_plots import plot_multichannel_timeseries

## 2. Generate Synthetic Signal

Create a synthetic neural signal that mimics the structure of real recordings
(multi-channel time series with noise).

In [ ]:
cfg = get_default_config()
np.random.seed(42)

n_timesteps = 200
n_channels = cfg.n_channels
fs = cfg.target_fs

# Simulate multi-channel neural data with band-limited activity + noise
t = np.arange(n_timesteps) / fs
signal = np.zeros((n_timesteps, n_channels))
for ch in range(n_channels):
    # Neural band activity (10-50 Hz oscillations)
    freq = np.random.uniform(10, 50)
    phase = np.random.uniform(0, 2 * np.pi)
    amplitude = np.random.uniform(0.5, 2.0)
    signal[:, ch] = amplitude * np.sin(2 * np.pi * freq * t + phase)
    # Add broadband noise
    signal[:, ch] += np.random.randn(n_timesteps) * 0.3

print(f"Synthetic signal shape: {signal.shape}")
print(f"Sampling rate: {fs} Hz")
print(f"Duration: {n_timesteps / fs:.2f} s")

## 3. Visualize Raw Signal

In [ ]:
fig = plot_multichannel_timeseries(
    signal[:, :8],
    title='Raw Synthetic Signal (first 8 channels)'
)
plt.show()

## 4. Preprocess

Apply the standard preprocessing pipeline: bandpass filter, notch filter,
and z-score normalization.

In [ ]:
# Step 1: Bandpass filter
filtered = bandpass_filter(signal, fs=fs)
print(f"After bandpass: {filtered.shape}")

# Step 2: Notch filter (60 Hz)
notched = notch_filter(filtered, fs=fs)
print(f"After notch: {notched.shape}")

# Step 3: Z-score normalization
normalized = zscore_normalize(notched)
print(f"After normalization: {normalized.shape}")
print(f"Mean of ch0: {normalized[:, 0].mean():.6f}, Std: {normalized[:, 0].std():.6f}")

## 5. Visualize Preprocessed Signal

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

for ch in range(min(5, n_channels)):
    axes[0].plot(t, signal[:, ch], linewidth=0.5, alpha=0.7)
axes[0].set_title('Raw Signal (5 channels)')
axes[0].set_xlabel('Time (s)')

for ch in range(min(5, n_channels)):
    axes[1].plot(t, normalized[:, ch], linewidth=0.5, alpha=0.7)
axes[1].set_title('Preprocessed Signal (5 channels)')
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

## 6. Model Inference

Load the GRU decoder and run a forward pass to get CTC log-probabilities.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GRUDecoder(
    input_size=cfg.n_channels,
    hidden_size=cfg.hidden_size,
    num_layers=cfg.num_layers,
    num_classes=cfg.num_classes,
).to(device)

# Load checkpoint if available
weight_path = os.path.join('..', 'checkpoints', 'gru_decoder_best.pt')
if os.path.exists(weight_path):
    model.load_state_dict(torch.load(weight_path, map_location=device))
    print("Loaded trained weights")
else:
    print("Using randomly initialized weights (demo mode)")

model.eval()

# Prepare input tensor: (batch=1, time, channels)
input_tensor = torch.tensor(normalized, dtype=torch.float32).unsqueeze(0).to(device)
print(f"Input tensor shape: {input_tensor.shape}")

with torch.no_grad():
    log_probs = model(input_tensor)

print(f"Output log_probs shape: {log_probs.shape}")
print(f"  (batch, time_steps, num_classes={cfg.num_classes})")

## 7. Greedy Decode

Take the argmax at each timestep and collapse repeated characters.

In [ ]:
greedy_result = greedy_decode(log_probs[0])
print(f"Greedy decode result: '{greedy_result}'") 

## 8. Beam Search Decode

Use beam search to explore multiple hypotheses and find higher-probability decodings.

In [ ]:
beam_results = beam_search_decode(log_probs[0], beam_width=10)
print("Beam search hypotheses:")
if isinstance(beam_results, list):
    for rank, hyp in enumerate(beam_results[:5]):
        if isinstance(hyp, tuple):
            text, score = hyp
            print(f"  #{rank+1}: '{text}' (score: {score:.4f})")
        else:
            print(f"  #{rank+1}: '{hyp}'")
else:
    print(f"  Best: '{beam_results}'")

## 9. Language Model Correction

Apply KenLM-based rescoring to improve decoding accuracy using language priors.

In [ ]:
try:
    lm_result = lm_rescore(beam_results)
    print(f"LM-rescored result: '{lm_result}'")
except Exception as e:
    print(f"LM rescoring not available: {e}")
    print("(This is expected if KenLM is not installed or no LM is trained)")
    lm_result = greedy_result

## 10. Visualize CTC Probabilities

Plot the CTC output probability heatmap to see the model's character predictions
over time.

In [ ]:
# Convert log probs to probabilities for visualization
probs = torch.exp(log_probs[0]).cpu().numpy()

fig = plot_ctc_heatmap(probs, title='CTC Output Probabilities')
plt.show()

## 11. Final Result Summary

Compare all three decoding strategies side by side.

In [ ]:
print("=" * 60)
print("DECODING RESULTS SUMMARY")
print("=" * 60)
print(f"  Greedy decode:     '{greedy_result}'")
if isinstance(beam_results, list) and len(beam_results) > 0:
    best_beam = beam_results[0][0] if isinstance(beam_results[0], tuple) else beam_results[0]
    print(f"  Beam search (k=10): '{best_beam}'")
else:
    print(f"  Beam search (k=10): '{beam_results}'")
print(f"  LM-corrected:      '{lm_result}'")
print("=" * 60)
print()
print("Pipeline stages completed:")
print("  1. Synthetic signal generation")
print("  2. Bandpass + notch filtering")
print("  3. Z-score normalization")
print("  4. GRU model forward pass")
print("  5. Greedy CTC decoding")
print("  6. Beam search decoding")
print("  7. Language model rescoring")
print("  8. CTC probability visualization")

---

**End of demo.** For the interactive version, run the Streamlit dashboard:
```bash
streamlit run app/dashboard.py
```